In [0]:
CHECKS = {
"I1_no_overlapping_intervals": """
    SELECT count(*) FROM (
      SELECT known_to_ts,
             LEAD(known_from_ts) OVER (
               PARTITION BY cik, canonical_tag, ddate, qtrs, uom
               ORDER BY known_from_ts) AS next_from
      FROM hindsight_dev.gold.fact_fundamental_pit)
    WHERE next_from IS NOT NULL AND next_from < known_to_ts
""",
"I2_exactly_one_open_interval": """
    SELECT count(*) FROM (
      SELECT sum(CASE WHEN is_current THEN 1 ELSE 0 END) AS n_open
      FROM hindsight_dev.gold.fact_fundamental_pit
      GROUP BY cik, canonical_tag, ddate, qtrs, uom)
    WHERE n_open <> 1
""",
"I3_no_gaps_between_intervals": """
    SELECT count(*) FROM (
      SELECT known_to_ts,
             LEAD(known_from_ts) OVER (
               PARTITION BY cik, canonical_tag, ddate, qtrs, uom
               ORDER BY known_from_ts) AS next_from
      FROM hindsight_dev.gold.fact_fundamental_pit)
    WHERE next_from IS NOT NULL AND next_from <> known_to_ts
""",
"I4_known_from_matches_source_filing": """
    SELECT count(*)
    FROM hindsight_dev.gold.fact_fundamental_pit f
    JOIN hindsight_dev.silver.sub s ON f.source_adsh = s.adsh
    WHERE f.known_from_ts <> s.accepted_ts
""",
"I4b_no_zero_width_intervals": """
    SELECT count(*) FROM hindsight_dev.gold.fact_fundamental_pit
    WHERE known_from_ts >= known_to_ts
""",
"I6_revision_seq_contiguous": """
    SELECT count(*) FROM (
      SELECT max(revision_seq) AS mx, count(*) AS n
      FROM hindsight_dev.gold.fact_fundamental_pit
      GROUP BY cik, canonical_tag, ddate, qtrs, uom)
    WHERE mx <> n
""",
}

failed = []
for name, sql in CHECKS.items():
    n = spark.sql(sql).collect()[0][0]
    print(f"{'PASS' if n == 0 else 'FAIL'}  {name}: {n:,} violations")
    if n:
        failed.append(name)

assert not failed, f"Invariants failed: {failed}"
print("\nAll structural invariants passed.")

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
from decimal import Decimal
from collections import defaultdict

sys.path.insert(0, "/Workspace/Users/saraogi.u@northeastern.edu/Hindsight")
from src.hindsight.precedence import Fact, build_intervals

SAMPLE = 300
SAMPLE_TBL = "hindsight_dev.ops.i5_sample_keys"

spark.sql(f"""
  CREATE OR REPLACE TABLE {SAMPLE_TBL} AS
  SELECT cik, canonical_tag, ddate, qtrs, uom
  FROM hindsight_dev.gold.fact_fundamental_pit
  WHERE revision_seq > 1
  GROUP BY cik, canonical_tag, ddate, qtrs, uom
  ORDER BY rand(42) LIMIT {SAMPLE}
""")
n_keys = spark.table(SAMPLE_TBL).count()

raw = spark.sql(f"""
  SELECT s.cik, n.canonical_tag, n.ddate, n.qtrs, n.uom, n.value,
         n.adsh, s.accepted_ts, s.form
  FROM hindsight_dev.silver.num n
  JOIN hindsight_dev.silver.sub s ON n.adsh = s.adsh
  JOIN {SAMPLE_TBL} k
    ON s.cik = k.cik AND n.canonical_tag = k.canonical_tag
   AND n.ddate = k.ddate AND n.qtrs = k.qtrs AND n.uom = k.uom
  WHERE n.value IS NOT NULL
""").collect()

gold = spark.sql(f"""
  SELECT f.cik, f.canonical_tag, f.ddate, f.qtrs, f.uom, f.value,
         f.known_from_ts, f.known_to_ts
  FROM hindsight_dev.gold.fact_fundamental_pit f
  JOIN {SAMPLE_TBL} k
    ON f.cik = k.cik AND f.canonical_tag = k.canonical_tag
   AND f.ddate = k.ddate AND f.qtrs = k.qtrs AND f.uom = k.uom
""").collect()

print(f"raw={len(raw)}  gold={len(gold)}  keys={n_keys}")


def norm(ts):
    return ts.replace(tzinfo=None) if ts and ts.tzinfo else ts


def kf(r):
    return (int(r.cik), r.canonical_tag, str(r.ddate)[:10], int(r.qtrs), r.uom.strip())


by_key = defaultdict(list)
for r in raw:
    by_key[kf(r)].append(
        Fact(cik=int(r.cik), canonical_tag=r.canonical_tag, ddate=str(r.ddate)[:10],
             qtrs=int(r.qtrs), uom=r.uom.strip(), value=Decimal(str(r.value)),
             adsh=r.adsh, accepted_ts=norm(r.accepted_ts), form=r.form))

gold_by_key = {}
for r in gold:
    gold_by_key.setdefault(kf(r), []).append(
        (Decimal(str(r.value)), norm(r.known_from_ts), norm(r.known_to_ts)))

mismatch, missing = [], []
for k, facts in by_key.items():
    if k not in gold_by_key:
        missing.append(k)
        continue
    oracle = [(i.value, i.known_from_ts, i.known_to_ts) for i in build_intervals(facts)]
    actual = sorted(gold_by_key[k], key=lambda x: x[1])
    if oracle != actual:
        mismatch.append((k, oracle, actual))

print(f"I5: {len(by_key)} keys | {len(mismatch)} mismatches | {len(missing)} missing")
for m in mismatch[:3]:
    print("\nKEY   ", m[0]); print("ORACLE", m[1]); print("SPARK ", m[2])
for k in missing[:3]:
    print("\nMISSING FROM GOLD:", k)

assert not mismatch and not missing, "I5 FAILED"
print("I5 passed -- Spark agrees with the tested oracle.")

In [0]:
print("raw rows   :", len(raw))
print("gold rows  :", len(gold))
print("sample keys:", len(key_rows))

if gold:
    g = gold[0]
    print("gold ddate:", type(g.ddate), repr(g.ddate), "| qtrs:", type(g.qtrs), "| uom:", repr(g.uom))
if raw:
    r = raw[0]
    print("raw  ddate:", type(r.ddate), repr(r.ddate), "| qtrs:", type(r.qtrs), "| uom:", repr(r.uom))

print("oracle key:", list(by_key.keys())[0] if by_key else "EMPTY")
print("gold   key:", list(gold_by_key.keys())[0] if gold_by_key else "EMPTY")

In [0]:
ok = set(by_key.keys())
gk = set(gold_by_key.keys())

print("oracle keys:", len(ok))
print("gold   keys:", len(gk))
print("intersection:", len(ok & gk))
print("in oracle only:", len(ok - gk))
print("in gold only  :", len(gk - ok))

if ok - gk:
    a = sorted(ok - gk)[0]
    print("\nmissing from gold:", a)
    # find the closest gold key on cik alone
    near = [k for k in gk if k[0] == a[0]]
    print("gold keys with same cik:", near[:5])
    if near:
        b = near[0]
        for i, (x, y) in enumerate(zip(a, b)):
            print(f"  pos {i}: oracle={x!r} ({type(x).__name__})  "
                  f"gold={y!r} ({type(y).__name__})  equal={x == y}")

In [0]:
k = sorted(ok & gk)[0]
print("KEY:", k)
print("\nORACLE:")
for iv in build_intervals(by_key[k]):
    print("  ", iv.value, iv.known_from_ts, "->", iv.known_to_ts)
print("\nGOLD (raw, unsorted):")
for row in gold_by_key[k]:
    print("  ", row)
print("\nRAW FACTS feeding oracle:")
for f in sorted(by_key[k], key=lambda x: x.accepted_ts):
    print("  ", f.value, f.accepted_ts, f.adsh)